# Agentic Multi-Hop RAG — Colab Experiment
**Before running:** `Runtime → Change runtime type → A100 GPU`

In [ ]:
# Cell 1: Mount Drive and set project path
from google.colab import drive
drive.mount('/content/drive')

import os

# Adjust if your Drive path is different
PROJECT_PATH = '/content/drive/MyDrive/CS572/Final'

os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Cell 2: Install dependencies
!pip install -q -U transformers bitsandbytes accelerate
!pip install -q rank-bm25 datasets sentence-transformers python-dotenv
print('Done.')

In [ ]:
# Cell 3: HuggingFace login
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(os.path.join(PROJECT_PATH, '.env'))
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
print('Logged in to HuggingFace.')

In [ ]:
# Cell 4: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 5: Add project root to Python path
import sys, shutil

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Clear stale __pycache__ and cached modules
for root, dirs, _ in os.walk(PROJECT_PATH):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Path set and cache cleared.')

In [ ]:
# Cell 6: Patch AgenticRAG and run_experiment to enforce min 2 hops (bypasses Drive sync issues)
import src.run_experiment as _run_exp
from src.agent import AgenticRAG, AGENT_PROMPT
import json, re

# --- Patch 1: AgenticRAG.answer with min_hops=2 and dedup-based stopping ---
def _new_answer(self, question):
    accumulated_context = []
    all_retrieved_docs = []
    per_hop_docs = []
    sub_queries = [question]
    current_query = question

    for hop in range(1, self.max_hops + 1):
        raw_docs = self.retriever.retrieve(current_query, top_k=self.top_k * 2)
        reranked_docs = self.reranker.rerank(current_query, raw_docs, top_k=self.top_k)
        all_retrieved_docs.extend(reranked_docs)
        per_hop_docs.append({'query': current_query, 'docs': reranked_docs})

        new_context = '\n'.join(d['text'] for d in reranked_docs)
        accumulated_context.append(f'[Hop {hop} — Query: {current_query}]\n{new_context}')

        prompt = AGENT_PROMPT.format(
            context='\n\n'.join(accumulated_context),
            question=question,
            hop=hop,
            max_hops=self.max_hops,
        )
        raw_response = self._call_llm(prompt)

        try:
            text = re.sub(r'```(?:json)?\s*', '', raw_response).strip().rstrip('`').strip()
            match = re.search(r'\{.*\}', text, re.DOTALL)
            decision = json.loads(match.group() if match else text)
        except (json.JSONDecodeError, ValueError, AttributeError):
            if hop >= self.min_hops:
                decision = {'sufficient': True, 'final_answer': ''}
            else:
                decision = {'sufficient': False, 'sub_query': question}

        next_query = decision.get('sub_query') or question
        # Also stop if LLM is proposing the same query again (no new info to retrieve)
        sufficient = decision.get('sufficient', False) or (
            hop >= self.min_hops and next_query.strip() == current_query.strip()
        )

        if sufficient and hop >= self.min_hops:
            final_answer = decision.get('final_answer', '')
            if not final_answer:
                context_str = '\n\n'.join(accumulated_context)
                final_answer = self._call_llm(
                    f'Answer concisely based on the context below.\n\n'
                    f'Context:\n{context_str}\n\nQuestion: {question}\n\nAnswer:'
                )
            return {
                'answer': final_answer,
                'num_hops': hop,
                'retrieved_docs': all_retrieved_docs,
                'per_hop_docs': per_hop_docs,
                'sub_queries': sub_queries,
            }

        sub_queries.append(next_query)
        current_query = next_query

    context_str = '\n\n'.join(accumulated_context)
    final_answer = self._call_llm(
        f'Answer concisely based on the context below.\n\n'
        f'Context:\n{context_str}\n\nQuestion: {question}\n\nAnswer:'
    )
    return {
        'answer': final_answer,
        'num_hops': self.max_hops,
        'retrieved_docs': all_retrieved_docs,
        'per_hop_docs': per_hop_docs,
        'sub_queries': sub_queries,
    }

AgenticRAG.answer = _new_answer
print('Patch 1: AgenticRAG.answer patched (min_hops=2, dedup stopping)')

# --- Patch 2: run_experiment._agent_answer_with_fallback ---
def _new_agent_answer_with_fallback(agent, question):
    try:
        return agent.answer(question)
    except Exception:
        reranked_docs = _run_exp._fallback_docs(agent, question)
        return {
            'answer': reranked_docs[0]['text'] if reranked_docs else '',
            'num_hops': 1,
            'retrieved_docs': reranked_docs,
            'per_hop_docs': [{'query': question, 'docs': reranked_docs}],
            'sub_queries': [question],
        }

_run_exp._agent_answer_with_fallback = _new_agent_answer_with_fallback
print('Patch 2: run_experiment._agent_answer_with_fallback patched (always calls agent.answer)')

In [ ]:
# Cell 7: Pilot run — 5 samples
from src.run_experiment import run

pilot_results = run(num_samples=5)
print('Pilot results:', pilot_results)

In [ ]:
# Cell 8: Full experiment — 50 samples (~60-90 min)
# Only run after pilot succeeds and avg_hops > 1.0 for agentic
from src.run_experiment import run

results = run(num_samples=50)
print('Final results:', results)

In [ ]:
# Cell 9: Confirm results saved to Drive
import json

results_path = os.path.join(PROJECT_PATH, 'results', 'experiment_results.json')
with open(results_path) as f:
    saved = json.load(f)

print('Baseline:', saved['baseline'])
print('Agentic: ', saved['agentic'])
print()
print('Saved to Drive at:', results_path)